# 🐸 Wednesday Frog – Flow Matching Training (Kaggle / Colab)

Zwei Stufen: **1. Pretrain** auf Froschfotos + Memes (lernt, wie Frösche aussehen), **2. Fine-Tune** nur auf die Memes.

**Kaggle:** Settings → Accelerator = *GPU T4 x2* oder *P100*, Internet = *On*. Unter *Add Input* zwei eigene Datasets hinzufügen:
- `wednesday-frogs` ← `data/wednesday-frogs.zip`
- `frog-photos` ← `data/pretrain/frog-photos.zip`

**Colab:** Runtime → GPU. Beide Zips in Google Drive (`MyDrive/`) legen.

Bei Timeout einfach alles nochmal ausführen – `train.py` setzt automatisch bei `last.pt` fort (sofern das Output-Verzeichnis erhalten bleibt, z.B. auf Drive).

In [ ]:
REPO = "https://github.com/alexanderulbrich/wednesday.git"
!git clone -q $REPO wednesday || (cd wednesday && git pull -q)
%cd wednesday
!pip install -q -e .
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import glob, os

def find_dataset(name):
    """Folder that contains the 128/ images of a dataset, wherever Kaggle/Colab put it."""
    hits = [d for d in glob.glob(f"{ROOT}/**/128", recursive=True) if name in d]
    if not hits:
        raise FileNotFoundError(f"Dataset '{name}' nicht gefunden unter {ROOT} – als Input hinzugefügt?")
    return hits[0]

if os.path.exists("/kaggle"):
    ROOT, OUT = "/kaggle/input", "/kaggle/working"
else:  # Colab
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT, OUT = "/content/data", "/content/drive/MyDrive/wednesday-runs"   # Runs auf Drive -> überleben Timeouts
    for z in ("wednesday-frogs", "frog-photos"):
        !mkdir -p {ROOT}/{z} && unzip -qo /content/drive/MyDrive/{z}.zip -d {ROOT}/{z}

MEMES, PHOTOS = find_dataset("wednesday-frogs"), find_dataset("frog-photos")
RES = 64   # trainiert wird in 64 px, Quelle sind die schärferen 128er Bilder
for d in (MEMES, PHOTOS):
    print(d, len(os.listdir(d)), "Bilder")

## Stufe 1: Pretrain (Froschfotos + Memes)

In [ ]:
!python train.py --data {PHOTOS},{MEMES} --res {RES} --out {OUT}/pre --steps 40000 --batch 64 --sample-every 2000 --save-every 5000 --workers 2

## Stufe 2: Fine-Tune (nur Memes)
Kürzer und mit kleinerer Lernrate. In den Sample-Rastern schauen, ab wann die Bilder nur noch Trainingsbilder kopieren – dann lieber einen früheren Stand nehmen.

In [ ]:
!python train.py --init {OUT}/pre/last.pt --data {MEMES} --res {RES} --out {OUT}/ft --steps 10000 --lr 1e-4 --warmup 200 --batch 64 --sample-every 1000 --save-every 2000 --workers 2

## Fortschritt anschauen

In [ ]:
from IPython.display import Image, display
RUN = "ft"   # oder "pre"
for s in sorted(glob.glob(f"{OUT}/{RUN}/samples/*.png"))[-3:]:
    print(s); display(Image(s))

Am Ende `frog.pt` (nur EMA-Gewichte, klein) aus dem Output herunterladen und auf dem Mac `python generate.py --ckpt frog.pt` ausführen.

In [ ]:
# Zwischenstand als frog.pt exportieren (falls ein Training noch nicht fertig ist)
import torch
RUN = "ft"   # oder "pre"
ck = torch.load(f"{OUT}/{RUN}/last.pt", map_location="cpu", weights_only=False)
torch.save({"ema": ck["ema"], "model_config": ck["model_config"], "step": ck["step"]}, f"{OUT}/{RUN}/frog.pt")
print("step", ck["step"], "->", f"{OUT}/{RUN}/frog.pt")